# Instruction Fine-Tuning Large Language Models for Summarization

This companion notebook contains the complete code workflow for the To Data & Beyond tutorial. It compares full fine-tuning and parameter-efficient LoRA fine-tuning of FLAN-T5 Base on DialogSum.

## Runtime notes

- Use a CUDA GPU runtime for meaningful training.
- The model and dataset are downloaded from Hugging Face; no credentials are required for these public resources.
- Training is deliberately limited to a small dataset subset and one optimization step so the tutorial remains approachable. Increase these settings for a real experiment.
- This notebook contains no saved execution outputs. Run cells in order.


## 1. Install and import dependencies


In [ ]:
%pip install -q -U transformers datasets evaluate rouge_score peft accelerate sentencepiece

In [ ]:
import time

import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    GenerationConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

## 2. Load DialogSum and FLAN-T5 Base


In [ ]:
dataset_name = "knkarthick/dialogsum"
model_name = "google/flan-t5-base"

dataset = load_dataset(dataset_name)
dataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_name)
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=model_dtype,
).to(device)
full_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=model_dtype,
).to(device)

In [ ]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    all_model_params = sum(parameter.numel() for parameter in model.parameters())
    percentage = 100 * trainable_model_params / all_model_params
    return (
        f"trainable model parameters: {trainable_model_params:,}\n"
        f"all model parameters: {all_model_params:,}\n"
        f"percentage of trainable model parameters: {percentage:.2f}%"
    )

print(print_number_of_trainable_model_parameters(full_model))

## 3. Establish a zero-shot baseline


In [ ]:
index = 200
dialogue = dataset["test"][index]["dialogue"]
human_summary = dataset["test"][index]["summary"]

prompt = f"""Summarize the following conversation.

{dialogue}

Summary:"""

inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    output_ids = baseline_model.generate(**inputs, max_new_tokens=200)
zero_shot_summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("INPUT PROMPT:\n", prompt)
print("\nBASELINE HUMAN SUMMARY:\n", human_summary)
print("\nMODEL GENERATION - ZERO SHOT:\n", zero_shot_summary)

## 4. Preprocess the dialogue-summary dataset


In [ ]:
def tokenize_function(examples):
    prompts = [
        f"Summarize the following conversation.\n\n{dialogue}\n\nSummary:"
        for dialogue in examples["dialogue"]
    ]
    model_inputs = tokenizer(prompts, max_length=512, truncation=True)
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=128,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

# Keep the source tutorial's small demonstration subset.
tokenized_datasets = tokenized_datasets.filter(
    lambda _, index: index % 100 == 0,
    with_indices=True,
)
tokenized_datasets

In [ ]:
print("Shapes of the datasets:")
print("Training:", tokenized_datasets["train"].shape)
print("Validation:", tokenized_datasets["validation"].shape)
print("Test:", tokenized_datasets["test"].shape)

## 5. Perform full fine-tuning


In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=full_model)
full_output_dir = f"./dialogue-summary-full-{int(time.time())}"

full_training_args = Seq2SeqTrainingArguments(
    output_dir=full_output_dir,
    learning_rate=1e-5,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=1,
    max_steps=1,
    save_strategy="no",
    report_to="none",
)

full_trainer = Seq2SeqTrainer(
    model=full_model,
    args=full_training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [ ]:
full_trainer.train()

In [ ]:
full_model_dir = "./full-dialogue-summary-model"
full_trainer.save_model(full_model_dir)
tokenizer.save_pretrained(full_model_dir)

trained_model = AutoModelForSeq2SeqLM.from_pretrained(
    full_model_dir,
    torch_dtype=model_dtype,
).to(device)

## 6. Evaluate the fully fine-tuned model


In [ ]:
generation_config = GenerationConfig(max_new_tokens=200, num_beams=1)

def generate_summary(model, dialogue_text):
    model_prompt = f"""Summarize the following conversation.

{dialogue_text}

Summary:"""
    model_inputs = tokenizer(model_prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        model_outputs = model.generate(
            **model_inputs,
            generation_config=generation_config,
        )
    return tokenizer.decode(model_outputs[0], skip_special_tokens=True)

baseline_text = generate_summary(baseline_model, dialogue)
full_text = generate_summary(trained_model, dialogue)

print("BASELINE HUMAN SUMMARY:\n", human_summary)
print("\nORIGINAL MODEL:\n", baseline_text)
print("\nFULLY FINE-TUNED MODEL:\n", full_text)

In [ ]:
rouge = evaluate.load("rouge")
dialogues = dataset["test"][:10]["dialogue"]
human_baseline_summaries = dataset["test"][:10]["summary"]

original_model_summaries = [
    generate_summary(baseline_model, item) for item in dialogues
]
full_model_summaries = [
    generate_summary(trained_model, item) for item in dialogues
]

full_comparison = pd.DataFrame(
    {
        "human_baseline_summaries": human_baseline_summaries,
        "original_model_summaries": original_model_summaries,
        "full_model_summaries": full_model_summaries,
    }
)
full_comparison

In [ ]:
original_model_results = rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries,
    use_aggregator=True,
    use_stemmer=True,
)
full_model_results = rouge.compute(
    predictions=full_model_summaries,
    references=human_baseline_summaries,
    use_aggregator=True,
    use_stemmer=True,
)

print("ORIGINAL MODEL:", original_model_results)
print("FULLY FINE-TUNED MODEL:", full_model_results)

## 7. Configure and train a PEFT/LoRA adapter


In [ ]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

peft_base_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=model_dtype,
)
peft_model = get_peft_model(peft_base_model, lora_config).to(device)
print(print_number_of_trainable_model_parameters(peft_model))

In [ ]:
peft_output_dir = f"./dialogue-summary-peft-{int(time.time())}"
peft_training_args = Seq2SeqTrainingArguments(
    output_dir=peft_output_dir,
    learning_rate=1e-3,
    num_train_epochs=1,
    logging_steps=1,
    max_steps=1,
    save_strategy="no",
    report_to="none",
)

peft_trainer = Seq2SeqTrainer(
    model=peft_model,
    args=peft_training_args,
    train_dataset=tokenized_datasets["train"],
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=peft_model),
    processing_class=tokenizer,
)

In [ ]:
peft_trainer.train()

peft_model_path = "./peft-dialogue-summary-adapter"
peft_trainer.model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)

In [ ]:
peft_inference_base = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=model_dtype,
)
peft_inference_model = PeftModel.from_pretrained(
    peft_inference_base,
    peft_model_path,
    is_trainable=False,
).to(device)

## 8. Evaluate the PEFT model


In [ ]:
peft_text = generate_summary(peft_inference_model, dialogue)

print("BASELINE HUMAN SUMMARY:\n", human_summary)
print("\nORIGINAL MODEL:\n", baseline_text)
print("\nFULLY FINE-TUNED MODEL:\n", full_text)
print("\nPEFT MODEL:\n", peft_text)

In [ ]:
peft_model_summaries = [
    generate_summary(peft_inference_model, item) for item in dialogues
]

peft_comparison = pd.DataFrame(
    {
        "human_baseline_summaries": human_baseline_summaries,
        "original_model_summaries": original_model_summaries,
        "full_model_summaries": full_model_summaries,
        "peft_model_summaries": peft_model_summaries,
    }
)
peft_comparison

In [ ]:
peft_model_results = rouge.compute(
    predictions=peft_model_summaries,
    references=human_baseline_summaries,
    use_aggregator=True,
    use_stemmer=True,
)

print("ORIGINAL MODEL:", original_model_results)
print("FULLY FINE-TUNED MODEL:", full_model_results)
print("PEFT MODEL:", peft_model_results)

In [ ]:
improvement_over_original = (
    np.array(list(peft_model_results.values()))
    - np.array(list(original_model_results.values()))
)
print("Absolute percentage improvement of PEFT MODEL over ORIGINAL MODEL")
for key, value in zip(peft_model_results, improvement_over_original):
    print(f"{key}: {value * 100:.2f}%")

In [ ]:
difference_from_full = (
    np.array(list(peft_model_results.values()))
    - np.array(list(full_model_results.values()))
)
print("Absolute percentage difference of PEFT MODEL from FULLY FINE-TUNED MODEL")
for key, value in zip(peft_model_results, difference_from_full):
    print(f"{key}: {value * 100:.2f}%")

## Next steps

The one-step training settings reproduce the article's compact teaching experiment, not a production training run. For a serious comparison, train on larger splits, tune the learning rate and LoRA rank, evaluate multiple checkpoints, and inspect summaries manually alongside ROUGE.
